In [ ]:
import os
import numpy as np
import pandas as pd
from itertools import permutations
import mygene

import sys
sys.path.insert(0, "../../packages/ProMERGE_utils")
from promerge_utils import uniprot_mapping

In [ ]:
os.makedirs("../../result/reference/", exist_ok=True)

# PPI reference

### ProHD2

In [ ]:
df = pd.read_csv("../../data/ProHD2/out/datasets/RF_04_disease_interactors_website", sep=",")
genes = sorted(set.union(set(df['Uniprot']), set(df['Interactor'])))

In [ ]:
# run uniprot ID mapping
mapping_dict = uniprot_mapping(genes, from_db="UniProtKB_AC-ID", to_db="Gene_Name")

In [ ]:
df['Uniprot'] = df['Uniprot'].map(mapping_dict)
df['Interactor'] = df['Interactor'].map(mapping_dict)
df = df.dropna()

geneA = df[['Uniprot', 'Interactor']].min(axis=1)
geneB = df[['Uniprot', 'Interactor']].max(axis=1)
df['Uniprot'] = geneA
df['Interactor'] = geneB
df = df[df['Uniprot'] != df['Interactor']]
df = df.sort_values('RF_covariation_prob', ascending=False)
df = df.drop_duplicates(subset=['Uniprot', 'Interactor'])

df.to_csv("../../result/reference/ProHD2_covariation.txt", sep="\t", index=False)

### PROPER

In [ ]:
proper = pd.read_csv("../../data/PROPER/PROPER_v1.csv")

# already BH-significant, keep explicit filtering anyway
proper = proper[proper["BH-corrected p-value"] < 0.05]
proper = proper[proper["Potential background contamination"] == 0]

geneA = proper[["Gene1", "Gene2"]].min(axis=1)
geneB = proper[["Gene1", "Gene2"]].max(axis=1)
proper["GeneA"] = geneA
proper["GeneB"] = geneB
proper = proper[proper["GeneA"] != proper["GeneB"]]
proper = proper.sort_values("Odds ratio", ascending=False)
proper = proper.drop_duplicates(subset=["GeneA", "GeneB"])
proper = proper.rename(columns={"Odds ratio": "Score"})

proper_hek = proper[proper["Cell line specificity"].isin(["HEK", "shared"])]
proper_share = proper[proper["Cell line specificity"].isin(["shared"])]
COLUMNS = ["GeneA", "GeneB", "Score"]
proper_share[COLUMNS].to_csv("../../result/reference/PROPER_share.txt", sep="\t", header=True, index=False)
proper_hek[COLUMNS].to_csv("../../result/reference/PROPER_HEK293T.txt", sep="\t", header=True, index=False)

### Image

In [ ]:
img = pd.read_csv("../../data/OpenCell/image_emd_1024pcs.tsv", sep="\t", index_col=0)

corr_matrix = np.corrcoef(img.values, rowvar=True)
corr_matrix = pd.DataFrame(corr_matrix, index=img.index.values, columns=img.index.values)
corr_long = corr_matrix.stack().reset_index()
corr_long.columns = ['GeneA', 'GeneB', 'Score']
corr_long = corr_long[corr_long['GeneA']<corr_long['GeneB']]

# keep top 1% as true pair
threshold = np.percentile(corr_long['Score'].abs(), 99)
corr_long = corr_long[corr_long['Score']>=threshold]
corr_long.to_csv("../../result/reference/ImageTop1.txt", sep="\t", header=True, index=False)

# Differential reference

### diff gene expression HPA

In [ ]:
RNA_LOG2FC = 2
RNA_MIN_NTPM = 1
RNA_CELL_LINE = {"HEK293": "HEK293", "HCT116": "HCT 116"}

chunks = pd.read_csv(
    "../../data/Expression/rna_celline.tsv.zip",
    sep="\t",
    usecols=["Gene name", "Cell line", "nTPM"],
    chunksize=2_000_000,
)
rna = pd.concat(
    [chunk[chunk["Cell line"].isin(RNA_CELL_LINE.values())] for chunk in chunks]
)
rna = rna.dropna(subset=["Gene name"])

# gene-level is the sum of all
rna = rna.groupby(["Gene name", "Cell line"])["nTPM"].sum().unstack().dropna()
rna["Log2ratio"] = np.log2(
    (rna[RNA_CELL_LINE["HEK293"]] + 1) / (rna[RNA_CELL_LINE["HCT116"]] + 1)
)

expressed = rna[list(RNA_CELL_LINE.values())].max(axis=1) >= RNA_MIN_NTPM
gold_standard = set(rna.index[expressed & (rna["Log2ratio"].abs() >= RNA_LOG2FC)])
with open("../../result/reference/HEK293T-HCT116_RNA.genes", 'w') as f:
    f.write("\n".join(sorted(gold_standard)))

with open("../../result/reference/HEK293T-HCT116_RNA.background", 'w') as f:
    f.write("\n".join(sorted(rna.index)))

### diff abundance (Zuniga)

In [ ]:
from scipy import stats

ZUNIGA_ADJ_P = 0.001
# MS2 acquisition compresses ratios: log2FC here spans ~0.58x the
# BioPlex comparison above, so 1 here matches ~2 there
ZUNIGA_LOG2FC = 1
ZUNIGA_REPLICATES = "ABCDEFGH"

raw = pd.read_excel(
    "../../data/TMT_Zuniga2024/pr4c00668_si_002.xlsx",
    header=None,
)
# rows 3-5 give sub-plex, cell line and replicate of each channel,
# proteins start at row 7 with the gene symbol in column 1
zuniga = raw.iloc[7:, 5:].astype(float)
zuniga.columns = raw.iloc[3:6, 5:].astype(str).agg("_".join)
zuniga.index = raw.iloc[7:, 1]
zuniga = zuniga[zuniga.index.notna() & (zuniga > 0).all(axis=1)]
log_sn = np.log2(zuniga)

# each lysate was split across the TMT and TMTD sub-plexes, so the
# two channels are averaged into one biological replicate
replicate = {
    cell: np.column_stack([
        (log_sn[f"TMT_{cell}_{r}"] + log_sn[f"TMTD_{cell}_{r}"]) / 2
        for r in ZUNIGA_REPLICATES
    ])
    for cell in ["HEK", "HCT"]
}
pvalue = stats.ttest_ind(
    replicate["HEK"],
    replicate["HCT"],
    axis=1,
    equal_var=False,
).pvalue
df = pd.DataFrame(
    {
        "Log2ratio": (
            replicate["HEK"].mean(axis=1)
            - replicate["HCT"].mean(axis=1)
        ),
        "AdjPValue": stats.false_discovery_control(pvalue),
    },
    index=log_sn.index,
)

# Keep most significant row if duplicate exist
df = df.sort_values("AdjPValue")
df = df[~df.index.duplicated(keep="first")]

gold_standard = set(df.index[
    (df["AdjPValue"] <= ZUNIGA_ADJ_P)
    & (df["Log2ratio"].abs() >= ZUNIGA_LOG2FC)
])
print(f"{len(gold_standard)} differential of {len(df)} genes")
with open(
    "../../result/reference/HEK293T-HCT116_abundanceTMT.genes",
    "w",
) as f:
    f.write("\n".join(sorted(gold_standard)))

with open(
    "../../result/reference/HEK293T-HCT116_abundanceTMT.background",
    "w",
) as f:
    f.write("\n".join(sorted(df.index)))

### stress (An 2020)

In [ ]:
from scipy import stats

STRESS_ADJ_P = 0.05
STRESS_LOG2DIFF = 0.3
STRESS_SOURCES = {
    "HEK293T": {
        "file": "1-s2.0-S1097276519302588-mmc2.xlsx",
        "sheet": "HA-6-80-1",
        "header": 2,
        "is_log2": True,
        "UT": [f"WT_UT{i}_ratio" for i in range(1, 4)],
        "AA": [f"WT_AA{i}_ratio" for i in range(1, 5)],
        "Torin1": [f"WT_Torin{i}_ratio" for i in range(1, 5)],
    },
    "HCT116": {
        "file": "41586_2020_2446_MOESM4_ESM.xlsx",
        "sheet": "HCT116",
        "header": 3,
        "is_log2": False,
        "UT": [f"WT_UT0{i}_sn sum" for i in range(1, 4)],
        "AA": [f"WT_AA0{i}_sn sum" for i in range(1, 5)],
        "Torin1": [f"WT_Tor0{i}_sn sum" for i in range(1, 5)],
    },
}
STRESS_GROUPS = ["UT", "AA", "Torin1"]

stress = {}
for cell, source in STRESS_SOURCES.items():
    sheet = pd.read_excel(
        f"../../data/Stress/{source['file']}",
        sheet_name=source["sheet"],
        header=source["header"],
        index_col="Gene Symbol",
    )
    values = sheet[sum([source[g] for g in STRESS_GROUPS], [])]
    values = values.astype(float).dropna()
    if not source["is_log2"]:
        # zeros are proteins quantified only in the knockout lines
        values = np.log2(values[(values > 0).all(axis=1)])
    values = values[~values.index.duplicated(keep="first")]
    stress[cell] = {g: values[source[g]] for g in STRESS_GROUPS}
shared = stress["HEK293T"]["UT"].index.intersection(
    stress["HCT116"]["UT"].index
)

gold_standard = set()
for treatment in ["AA", "Torin1"]:
    # (HEK stress - HEK UT) - (HCT stress - HCT UT), Welch-type test
    # with Satterthwaite degrees of freedom over the four groups
    diff = 0
    var_terms = []
    n_terms = []
    for cell, sign in [("HEK293T", 1), ("HCT116", -1)]:
        for group, weight in [(treatment, 1), ("UT", -1)]:
            values = stress[cell][group].loc[shared]
            n = values.shape[1]
            diff = diff + sign * weight * values.mean(axis=1)
            var_terms.append(values.var(axis=1, ddof=1) / n)
            n_terms.append(n)
    se2 = sum(var_terms)
    dof = se2 ** 2 / sum(
        v ** 2 / (n - 1) for v, n in zip(var_terms, n_terms)
    )
    pvalue = 2 * stats.t.sf(np.abs(diff / np.sqrt(se2)), dof)
    adj_p = stats.false_discovery_control(pvalue)
    hits = set(shared[
        (adj_p <= STRESS_ADJ_P)
        & (np.abs(diff) >= STRESS_LOG2DIFF)
    ])
    print(f"{treatment}: {len(hits)} differential")
    gold_standard |= hits

print(f"{len(gold_standard)} differential of {len(shared)} genes")
with open(
    "../../result/reference/HEK293T-HCT116_stress.genes",
    "w",
) as f:
    f.write("\n".join(sorted(gold_standard)))

with open(
    "../../result/reference/HEK293T-HCT116_stress.background",
    "w",
) as f:
    f.write("\n".join(sorted(shared)))